# Harness Engineering Demo: Better Harness Beats Bigger Model

This notebook is built for a management-facing demo.

**Thesis:** model quality matters, but harness maturity determines whether an AI workflow is production-ready. A medium model in a strong harness can outperform a stronger model in a no-harness or weak-harness setup because production quality depends on governed context, tools, validation, memory, safety gates, observability, and repair loops.

We will show the spectrum:

1. **Great model + no harness**: incident ticket only, no controlled guides, sensors, or steering loop.
2. **Great model + weak harness**: multi-agent scratchpad exists, but it is untyped, unprovenanced, and not governed by sensors or repair.
3. **Medium model + SDK harness**: Strands-style abstraction for tools, hooks, memory, and multi-agent orchestration.
4. **Provider plug-and-play harness**: provider-specific harness lane, represented by DeepSeek adapter boundary.

The reliable part of the demo is deterministic. The live model section calls Ollama Cloud so management can see how this connects to real model backends.

## Demo Architecture

```text
Colab notebook
      ↓
Harness demo repo + Python SDKs
      ↓
Scenario: multi-agent incident response
      ↓
Scorecard: evidence, runbook, safety, memory, completeness
      ↓
Optional live calls to Ollama Cloud
```

Colab is the runtime. Ollama Cloud is the model backend.

## 1. Clone The Repo

Replace `REPO_URL` with your GitHub URL after pushing the project.

If you already uploaded this notebook into the cloned repo, skip this cell and `%cd` into the repo folder.

In [ ]:
# Replace this with your pushed GitHub repository URL.
REPO_URL = "https://github.com/YOUR_ORG/ollama-harness-engineering-demo.git"
REPO_DIR = "ollama-harness-engineering-demo"

from pathlib import Path
import os

# If we are not already inside the repo, clone it or move into an existing clone.
if not Path("pyproject.toml").exists():
    if Path(REPO_DIR).exists():
        os.chdir(REPO_DIR)
    else:
        if "YOUR_ORG" in REPO_URL:
            raise ValueError(
                "Replace REPO_URL with your GitHub repo URL, then rerun this cell. "
                "Example: https://github.com/my-org/ollama-harness-engineering-demo.git"
            )
        !git clone $REPO_URL
        os.chdir(REPO_DIR)

print("Current directory:", Path.cwd())
print("Project files:")
!ls -la

assert Path("requirements.txt").exists(), "requirements.txt not found. You are not inside the repo."
assert Path("pyproject.toml").exists(), "pyproject.toml not found. You are not inside the repo."

## 2. Install The Demo Dependencies

This happens inside Colab, so it does not depend on your office Mac allowing Python packages.

The important libraries are:

- `strands-agents`: SDK-level harness abstraction.
- `ollama`: direct calls to Ollama Cloud.
- `openai`: useful for OpenAI-compatible endpoints.
- `typer` and `rich`: CLI and readable scorecards.
- `pytest`: quick health check.

In [ ]:
from pathlib import Path

assert Path("requirements.txt").exists(), "Run the clone/%cd setup cell first. requirements.txt is missing here."
assert Path("pyproject.toml").exists(), "Run the clone/%cd setup cell first. pyproject.toml is missing here."

!pip install -r requirements.txt
!pip install -e .

## 3. Import The Repo Harness

Colab is only the presentation surface. The actual use case and harness workflow are imported from the repo.

No static lane outputs are used in this notebook. The management demo path below calls Ollama Cloud live and scores the actual model-generated artifacts.

In [ ]:
from pathlib import Path
from pprint import pprint
import sys

repo_src = str(Path.cwd() / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)

assert Path("src/harness_demo").exists(), "Not in repo root or src/harness_demo is missing."

from IPython.display import HTML, Markdown, display

from harness_demo.live import run_live_hand_built_lane, run_live_raw_lane, run_live_weak_harness_lane, score_freeform_answer
from harness_demo.rules import evaluate_rules
from harness_demo.summarizer import summarize_findings_with_ollama
from harness_demo.colab_display import (
    render_comparison_markdown,
    render_hallucination_review_markdown,
    render_management_summary_markdown,
    render_executive_findings_markdown,
    render_memory_markdown,
    render_model_output_html,
    render_result_markdown,
    render_rule_findings_markdown,
)
from harness_demo.scenarios import load_incident_scenario
from harness_demo.domain import Lane

scenario = load_incident_scenario("incident-response")

print("Loaded scenario:", scenario.id)
print("Scenario name:", scenario.name)

## 4. The Use Case: Incident Response, Not Prompt Comparison

The scenario is a production checkout incident after a promotion launch.

A management audience should see the difference between:

- a model answering from a ticket alone
- an AI workflow that uses controlled evidence, approved runbooks, shared memory, reviewer checks, and repair

This is the point of harness engineering: the system around the model turns an answer into a governed workflow result.

In [ ]:
print("INCIDENT TICKET")
pprint(scenario.incident)

print("\nQUALITY CONTRACT USED BY THE SCORER")
pprint(scenario.expected)

## 5. Controlled Context Available To The Harness

These are not pasted wholesale into the raw model call.

They are available to the **harnessed workflow** through separate controlled steps:

- log investigator sees logs
- runbook agent sees runbook
- memory agent sees prior incident memory
- planner sees the shared memory created by earlier agents
- reviewer sees the final plan and checks policy/completeness

That separation is the harness. It is what changes between the weak and strong setup.

In [ ]:
print("LOG TOOL DATA")
print(scenario.logs)

print("RUNBOOK TOOL DATA")
print(scenario.runbook)

print("PRIOR MEMORY TOOL DATA")
print(scenario.prior_memory)

## 6. Three Different Things: No Harness, Weak Harness, Strong Harness

We should be precise. Missing memory sharing is not a weak harness. It is mostly **no harness**.

For this demo we use three levels:

| Level | What it means in this incident workflow |
| --- | --- |
| No harness | One model call against the incident ticket. No explicit guides, no tools, no sensors, no steering loop. |
| Weak harness | Multi-agent workflow with a shared scratchpad. It has some feedforward context and memory sharing, but memory is untyped and unprovenanced, and there are no independent sensors before the final plan, no reviewer gate, and no repair loop. |
| Strong harness | Work is decomposed into controlled steps with explicit guides, tool-scoped context, shared memory, deterministic sensors, reviewer checks, and repair. |

This is closer to the harness-engineering framing: a harness is a system of **feedforward guides**, **feedback sensors**, and a **steering/self-correction loop**. A weak harness has only some of these pieces, or has them in a non-operational form.

## 7. What Changes In The Strong Harness?

The strong harness is not merely a longer prompt. It changes the execution environment around the model.

| Harness component | Strong harness behavior in this demo |
| --- | --- |
| Feedforward guides | Each agent gets a narrow role and only the context it is supposed to use. |
| Controlled tools | Logs, runbook, and prior memory are separate inputs to separate steps, not one undifferentiated paste. |
| Shared state | Agent outputs are accumulated into `SharedMemory`, which later steps consume. |
| Computational sensors | The scorer checks evidence, runbook usage, safety, memory usage, and completeness. |
| Reviewer gate | The final plan is checked for forbidden actions and missing required fields. |
| Steering loop | If the reviewer finds issues, the repair step asks the model to revise against concrete objections. |

The medium model does not win because it was prompted more nicely. It gets a more governable operating environment.

## 8. Configure Ollama Cloud

Ollama Cloud is the live backend. The repo harness is the application workflow.

Do not hardcode the key in the notebook.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OLLAMA_API_KEY"):
    os.environ["OLLAMA_API_KEY"] = getpass("Enter OLLAMA_API_KEY: ")

print("OLLAMA_API_KEY configured:", bool(os.environ.get("OLLAMA_API_KEY")))

## 9. Choose Models

The intended comparison:

- **strong model + no harness**: bare incident ticket only
- **strong model + weak harness**: multi-agent scratchpad and context, but memory is untyped/unprovenanced and there are no sensors or repair
- **medium model + strong harness**: decomposed workflow with guides, controlled context, memory, sensors, reviewer, and repair

Change these model names based on your Ollama Cloud subscription.

In [ ]:
NO_HARNESS_MODEL = "gpt-oss:120b"
WEAK_HARNESS_MODEL = "gpt-oss:120b"
STRONG_HARNESS_MODEL = "gpt-oss:20b"

print("Strong model, no harness:", NO_HARNESS_MODEL)
print("Strong model, weak harness:", WEAK_HARNESS_MODEL)
print("Medium model, strong harness:", STRONG_HARNESS_MODEL)

## Optional Summarizer Model

The deterministic rule engine decides truth. This model only rewrites findings into a management-friendly summary.

Use a cheaper/simple model here if available.

In [ ]:
SUMMARY_MODEL = "gpt-oss:20b"
print("Summary/polish model:", SUMMARY_MODEL)

# Live Run 1: Strong Model + No Harness

This cell makes one live Ollama Cloud call.

The model receives the incident ticket only. It does not receive logs, runbook, prior memory, output contract, reviewer, or repair loop.

This is the **no-harness baseline**, not a weak multi-agent harness.

In [ ]:
live_no_harness = run_live_raw_lane(scenario, model_name=NO_HARNESS_MODEL)
display(Markdown(render_management_summary_markdown(live_no_harness)))
display(Markdown(render_executive_findings_markdown(live_no_harness)))
display(Markdown(render_rule_findings_markdown(scenario, live_no_harness)))
display(HTML(render_model_output_html("Actual model output", live_no_harness.final_answer)))
display(Markdown(render_memory_markdown(live_no_harness.memory)))
display(Markdown(render_hallucination_review_markdown(live_no_harness)))


# Live Run 2: Strong Model + Weak Harness

This is the actual **weak harness** comparison.

The strong model runs in a weak multi-agent harness. There is shared memory, but it is only a plain scratchpad.

That is better than no harness because agents can pass context forward. But it is still weak because:

- the scratchpad has no schema
- memory entries have no source provenance
- the workflow does not validate intermediate notes
- the final plan is not reviewed before scoring
- there is no repair loop

This is the realistic corporate distinction: shared memory alone is not harness engineering. Governed memory plus sensors and steering is.

In [ ]:
live_weak = run_live_weak_harness_lane(scenario, model_name=WEAK_HARNESS_MODEL)
display(Markdown(render_management_summary_markdown(live_weak)))
display(Markdown(render_executive_findings_markdown(live_weak)))
display(Markdown(render_rule_findings_markdown(scenario, live_weak)))
display(HTML(render_model_output_html("Actual model output", live_weak.final_answer)))
display(Markdown(render_memory_markdown(live_weak.memory)))


# Live Run 3: Medium Model + Strong Harness

This cell calls Ollama Cloud multiple times through the repo harness workflow.

Each call has a specific role and a controlled context boundary:

1. log investigator: incident + logs
2. runbook agent: incident + runbook
3. memory agent: incident + prior memory
4. planner: shared memory + required output fields + forbidden actions
5. reviewer: deterministic safety/completeness checks
6. repair: only if reviewer finds issues

The score is computed from the actual generated agent outputs and final plan.

In [ ]:
live_harness = run_live_hand_built_lane(scenario, model_name=STRONG_HARNESS_MODEL)
display(Markdown(render_management_summary_markdown(live_harness)))
display(Markdown(render_executive_findings_markdown(live_harness)))
display(Markdown(render_rule_findings_markdown(scenario, live_harness)))
display(HTML(render_model_output_html("Actual agent outputs", live_harness.final_answer)))


## 10. Inspect The Shared Memory

This is the harness value in concrete form.

The workflow did not just produce prose. It produced operational state that can be checked, audited, and reused.

In [ ]:
display(Markdown(render_memory_markdown(live_harness.memory)))


## 11. Live Side-By-Side Result

This is the only scorecard to show as evidence.

All rows come from live Ollama Cloud calls. The difference is the harness maturity, not a hardcoded result.

In [ ]:
display(Markdown(render_comparison_markdown([live_no_harness, live_weak, live_harness])))


## 12. LLM-Polished Management Summaries

These summaries are generated by an Ollama Cloud model, but the model is **not judging**.

Inputs to the summarizer:

- deterministic score
- deterministic pass/fail checks
- deterministic rule findings
- extracted shared memory

The summarizer is only used to make the findings easier to present.

In [ ]:
for label, result in [
    ("No harness", live_no_harness),
    ("Weak harness", live_weak),
    ("Strong harness", live_harness),
]:
    findings = evaluate_rules(scenario, result)
    summary = summarize_findings_with_ollama(
        scenario=scenario,
        result=result,
        findings=findings,
        model_name=SUMMARY_MODEL,
    )
    display(Markdown(f"# {label}: LLM-Polished Summary"))
    display(Markdown(summary))

## 12. Explain The Difference In Plain English

Use this in the meeting after the live scorecard:

- The no-harness lane asks a strong model to improvise from the ticket alone.
- The weak-harness lane has multi-agent shared memory, but the memory is an ungoverned scratchpad with no schema, provenance, sensors, or steering.
- The strong-harness lane decomposes the work, gives each agent controlled context, accumulates shared memory, and checks the final result against policy.
- The scorecard is not grading writing style. It is grading production requirements: evidence, runbook use, safety, memory, completeness.

That is harness engineering.

# Optional Smoke Test

Only use this for your own pre-demo sanity check, not as a management proof.

```bash
harness-demo compare --scenario incident-response
python -m pytest -p no:cacheprovider
```

Those commands validate the repo wiring. The live evidence is above.

# Industry Spectrum: Harness Engineering Is Getting Productized

After the live evidence, show management that this is not only our custom pattern. The industry is moving from hand-built harnesses toward SDKs and plug-and-play harness runtimes.

Important boundary:

- The **proof** in this notebook is the live Ollama Cloud no/weak/strong harness comparison above.
- The **industry movement** section below explains why teams should expect more of these controls to become reusable infrastructure.

## Strands Agents: SDK-Level Harness Abstraction

Strands is the SDK-level part of the story.

It shows that common harness capabilities are becoming framework features:

| Harness need | Strands direction |
| --- | --- |
| Tool boundaries | `@tool` definitions and tool registries |
| Feedforward guides | agent instructions, tool descriptions, steering handlers |
| Feedback sensors | hooks before/after tool calls |
| Context management | conversation managers and summarization |
| Safety controls | guardrails/hooks that block or redirect actions |
| Observability | traces and hook-level inspection |
| Multi-agent workflows | agent-as-tool and swarm-style patterns |

Management message:

> We hand-built the harness to make the mechanics visible. SDKs like Strands show these mechanics are becoming reusable developer infrastructure.

Source: https://strandsagents.com/

In [ ]:
# Lightweight sanity check: Strands is installed in the Colab environment.
# This is not the live proof. It shows the SDK is available for the next implementation layer.
import strands
print("Strands SDK import ok:", strands.__name__)

## DeepSeek Harness: Plug-And-Play Harness Runtime

DeepSeek Harness is the plug-and-play runtime part of the story.

Its public developer-preview positioning is **“Everything is a plugin.”** The harness runtime composes capabilities such as:

- models
- tools
- skills
- sessions
- sandboxes
- storage
- loops
- scheduling
- UI

It also emphasizes traceability: what the model sees, tool calls/results, context injection, and subagent scheduling are recorded in an append-only session log.

Management message:

> DeepSeek Harness shows where the market is heading: harness capabilities are becoming runtime infrastructure that teams can compose instead of rebuilding from scratch.

Source: https://deepseek.com/harness/en/

## How To Say This Without Overselling

Use this wording:

> Today we proved the harness value with our live hand-built workflow. Strands and DeepSeek Harness show that the same ideas are being productized: SDKs are packaging tools, hooks, memory, and observability; harness runtimes are packaging models, tools, sessions, loops, and traceability as plugins.

Avoid this wording:

> Strands/DeepSeek produced the same scores in this notebook.

We should only make that claim after wiring them into live execution.

# Adoption Experiment 1: Strands Multi-Agent SDK Harness

This is the “team adoption” section, now structured as a fairer multi-agent comparison.

Goal: show that the controls we hand-built are becoming SDK-level concepts: separate agents, scoped tools, explicit handoffs, trace attributes, and model-provider abstraction.

This cell attempts a real Strands multi-agent run with the same incident scenario:

1. **Log agent** sees only logs.
2. **Runbook agent** sees only the approved runbook.
3. **Memory agent** sees only prior incident memory.
4. **Planner agent** receives the shared handoff from the specialist agents and writes the final plan.
5. The repo deterministic evaluator scores the final answer.

This is still not magic. Strands supplies SDK primitives; our harness contract still defines the roles, handoffs, and scoring gates.

Sources: https://strandsagents.com/docs/user-guide/concepts/tools/ and https://strandsagents.com/docs/user-guide/concepts/multi-agent/agents-as-tools/


In [ ]:
STRANDS_MODEL = "gpt-oss:20b"  # adjust if your Strands provider expects a different model id
print("Strands experiment model:", STRANDS_MODEL)

In [ ]:
strands_result = None
try:
    import os
    import json
    from strands import Agent, tool
    from strands.models.ollama import OllamaModel

    if not os.environ.get("OLLAMA_API_KEY"):
        raise RuntimeError("OLLAMA_API_KEY is required for the Strands Ollama Cloud lane.")

    strands_model = OllamaModel(
        host="https://ollama.com",
        model_id=STRANDS_MODEL,
        ollama_client_args={
            "headers": {"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]}
        },
        temperature=0.2,
    )

    @tool
    def get_checkout_logs() -> str:
        """Return checkout incident logs."""
        return scenario.logs

    @tool
    def get_checkout_runbook() -> str:
        """Return the approved checkout promotion incident runbook."""
        return scenario.runbook

    @tool
    def get_prior_incident_memory() -> str:
        """Return prior similar incident memory."""
        return scenario.prior_memory

    log_agent = Agent(
        model=strands_model,
        tools=[get_checkout_logs],
        system_prompt=(
            "You are the log investigator agent. Use only get_checkout_logs. "
            "Return evidence for likely cause and downstream symptoms. Do not invent tools or dashboards."
        ),
        trace_attributes={"demo": "harness-engineering", "lane": "strands-sdk", "agent": "log-investigator"},
    )
    runbook_agent = Agent(
        model=strands_model,
        tools=[get_checkout_runbook],
        system_prompt=(
            "You are the runbook agent. Use only get_checkout_runbook. "
            "Return approved mitigation steps and safety constraints. Do not add unapproved operations."
        ),
        trace_attributes={"demo": "harness-engineering", "lane": "strands-sdk", "agent": "runbook"},
    )
    memory_agent = Agent(
        model=strands_model,
        tools=[get_prior_incident_memory],
        system_prompt=(
            "You are the memory agent. Use only get_prior_incident_memory. "
            "Return prior lessons relevant to this incident."
        ),
        trace_attributes={"demo": "harness-engineering", "lane": "strands-sdk", "agent": "memory"},
    )

    log_output = str(log_agent(
        "Incident ticket:\n" + scenario.incident["prompt"] + "\n\nCall the log tool and return grounded evidence only."
    ))
    runbook_output = str(runbook_agent(
        "Incident ticket:\n" + scenario.incident["prompt"] + "\n\nCall the runbook tool and return approved actions and constraints only."
    ))
    memory_output = str(memory_agent(
        "Incident ticket:\n" + scenario.incident["prompt"] + "\n\nCall the prior memory tool and return relevant lessons only."
    ))

    strands_shared_memory = {
        "incident": scenario.incident,
        "log_agent_evidence": log_output,
        "runbook_agent_constraints": runbook_output,
        "memory_agent_lessons": memory_output,
        "required_output_fields": scenario.expected["required_final_plan_fields"],
        "forbidden_actions": scenario.expected["forbidden_actions"],
    }

    planner_agent = Agent(
        model=strands_model,
        system_prompt=(
            "You are the planner agent in a Strands multi-agent harness. "
            "Use the specialist handoff exactly. Return JSON or concise markdown with likely_cause, "
            "evidence, safe_next_action, rollback_plan, customer_impact, and open_questions. "
            "Do not invent tools, owners, dashboards, thresholds, or operational facts that are not in the handoff."
        ),
        trace_attributes={"demo": "harness-engineering", "lane": "strands-sdk", "agent": "planner"},
    )
    planner_output = str(planner_agent(json.dumps(strands_shared_memory, indent=2)))

    strands_output = "\n\n".join([
        "## Strands Log Agent",
        log_output,
        "## Strands Runbook Agent",
        runbook_output,
        "## Strands Memory Agent",
        memory_output,
        "## Strands Planner Agent",
        planner_output,
    ])

    strands_result = score_freeform_answer(
        scenario=scenario,
        answer=strands_output,
        lane=Lane.STRANDS_SDK,
        title="Strands multi-agent SDK harness experiment",
        takeaway=(
            "Strands now runs separate specialist agents with scoped tools and explicit handoffs. "
            "The repo deterministic evaluator still decides whether the output is production-ready."
        ),
        used_harness_memory=True,
    )

    # Add the same harness maturity pattern as the hand-built lane: deterministic review -> repair -> rescore.
    # Strands supplies SDK agents; our harness contract still supplies the quality gate.
    if not all(strands_result.checks.values()):
        strands_findings_for_repair = evaluate_rules(scenario, strands_result)
        repair_agent = Agent(
            model=strands_model,
            system_prompt=(
                "You are the repair agent in a Strands multi-agent production harness. "
                "Revise the final plan to satisfy deterministic reviewer findings. "
                "Use only the specialist handoff, approved runbook steps, and required fields. "
                "Return JSON only. Do not invent tools, owners, dashboards, thresholds, or operational facts."
            ),
            trace_attributes={"demo": "harness-engineering", "lane": "strands-sdk", "agent": "repair"},
        )
        repair_payload = {
            "deterministic_findings": [finding.__dict__ for finding in strands_findings_for_repair],
            "specialist_handoff": strands_shared_memory,
            "current_planner_output": planner_output,
            "required_output_fields": scenario.expected["required_final_plan_fields"],
            "required_evidence": scenario.expected["required_evidence"],
            "required_runbook_steps": scenario.expected["required_runbook_steps"],
            "forbidden_actions": scenario.expected["forbidden_actions"],
        }
        repair_output = str(repair_agent(json.dumps(repair_payload, indent=2)))
        strands_output = strands_output + "\n\n## Strands Repair Agent\n" + repair_output
        strands_result = score_freeform_answer(
            scenario=scenario,
            answer=strands_output,
            lane=Lane.STRANDS_SDK,
            title="Strands multi-agent SDK harness experiment with repair",
            takeaway=(
                "Strands runs specialist agents, deterministic review, and a repair agent. "
                "The repo deterministic evaluator still decides whether the repaired output is production-ready."
            ),
            used_harness_memory=True,
        )

    display(Markdown(render_management_summary_markdown(strands_result)))
    display(Markdown(render_executive_findings_markdown(strands_result)))
    display(Markdown(render_rule_findings_markdown(scenario, strands_result)))
    display(HTML(render_model_output_html("Strands multi-agent outputs", strands_result.final_answer)))
except Exception as exc:
    display(Markdown(f"""
## Strands Multi-Agent Experiment Did Not Produce A Scorable Output

This does **not** invalidate harness engineering. It means the Strands provider/runtime configuration needs more setup in this Colab environment.

**Error type:** `{type(exc).__name__}`  
**Error:** `{exc}`

What to try next:

- confirm `strands.models.ollama.OllamaModel` supports Ollama Cloud auth headers in this version
- confirm `OLLAMA_API_KEY` is set
- confirm `STRANDS_MODEL` is available in your Ollama Cloud subscription
- keep the hand-built harness as the proof and use Strands as the SDK adoption path
"""))


In [ ]:
if strands_result is not None:
    strands_findings = evaluate_rules(scenario, strands_result)
    strands_summary = summarize_findings_with_ollama(
        scenario=scenario,
        result=strands_result,
        findings=strands_findings,
        model_name=SUMMARY_MODEL,
    )
    display(Markdown("# Strands: LLM-Polished Summary"))
    display(Markdown(strands_summary))
else:
    display(Markdown("_No Strands summary generated because there was no scorable Strands output._"))

# Adoption Experiment 2: DeepSeek Harness + Ollama

DeepSeek Harness is a developer-preview plug-and-play harness runtime. For the Ollama-backed path, do **not** start with `npx @deepseek-ai/dsh --help` in Colab. That command can download packages and hang, and it does not prove the Ollama integration.

Use the official Ollama integration path instead:

```bash
ollama launch dsh
ollama launch dsh --model deepseek-v4-flash:cloud
ollama launch dsh --config
```

What this proves:

- Ollama is the provider/launcher boundary.
- DeepSeek Harness is the agent runtime/harness boundary.
- Ollama writes DeepSeek Harness launch settings under `~/.ollama/launch/dsh/settings.yaml`.
- Web search requires Ollama Cloud access and a tool-capable model.

This section therefore uses bounded commands and treats DeepSeek as an adoption experiment. If it runs, paste or capture the output and score it with the same deterministic evaluator. If it does not run cleanly in Colab, say that the developer-preview runtime path needs environment hardening; the harness-engineering proof remains the live no/weak/strong comparison above.

Source: https://docs.ollama.com/integrations/deepseek-harness


In [ ]:
# DeepSeek Harness + Ollama Cloud setup check.
# This intentionally avoids `npx @deepseek-ai/dsh --help` because that can hang in Colab
# and does not exercise the Ollama integration documented by Ollama.

import os
from pathlib import Path
from IPython.display import Markdown, display

DEEPSEEK_DSH_MODEL = os.environ.get("DEEPSEEK_DSH_MODEL", "deepseek-v4-flash:cloud")
os.environ["DEEPSEEK_DSH_MODEL"] = DEEPSEEK_DSH_MODEL

display(Markdown(f"""
## DeepSeek Harness Setup Check

Model selected for the Ollama launcher: `{DEEPSEEK_DSH_MODEL}`

The official integration command is:

```bash
ollama launch dsh --model {DEEPSEEK_DSH_MODEL}
```

For Colab safety, this cell only verifies the local tools and runs the non-starting config command with a timeout.
"""))

!timeout 20s node --version || true
!timeout 20s ollama --version || true

# If `ollama` is missing in Colab, install the Ollama CLI/runtime first.
# Run this manually only if the version check above says `ollama: command not found`:
# !timeout 180s sh -c 'curl -fsSL https://ollama.com/install.sh | sh'

# Cloud auth is required for cloud models. In a local terminal this is:
# !ollama signin
# In Colab this may require browser/auth handoff. Do it before launching DSH.

# Configure DeepSeek Harness through Ollama without starting the long-running web UI.
# The Ollama docs say this writes settings under ~/.ollama/launch/dsh/settings.yaml.
!timeout 90s ollama launch dsh --model "$DEEPSEEK_DSH_MODEL" --config || true

settings_path = Path.home() / ".ollama" / "launch" / "dsh" / "settings.yaml"
if settings_path.exists():
    display(Markdown("## Generated Ollama DSH Settings"))
    print(settings_path.read_text())
else:
    display(Markdown("""
## DeepSeek Harness Settings Not Created Yet

This usually means one of these is missing in the Colab runtime:

- Ollama CLI/runtime is not installed.
- `ollama signin` has not completed.
- The selected cloud model is not available to the account.
- The developer-preview launcher needs an interactive/local environment.

This is an adoption-path setup finding, not a failure of the harness engineering result already demonstrated above.
"""))


In [ ]:
# Optional: attempt a bounded DeepSeek Harness headless run through Ollama.
# This depends on the current DSH preview supporting `--profile headless` through the launcher.
# If it fails, keep the error as the framework maturity/setup evidence and use the paste/scoring cell below.

from pathlib import Path

dsh_workspace = Path("/content/dsh-incident-demo")
dsh_workspace.mkdir(exist_ok=True)

(dsh_workspace / "incident_ticket.md").write_text("""
Incident: INC-2026-08-17-042
Service: checkout-api
Detected: 2026-08-17T09:42:00Z
Impact: Customers see slow checkout and intermittent payment timeout errors.

Observed facts:
- p95 latency increased from 240ms to 2100ms.
- Logs show repeated promotion_price_cache miss events.
- Payment timeout errors appear downstream of checkout latency.
""".strip())

(dsh_workspace / "runbook.md").write_text("""
Runbook:
- Treat promotion_price_cache miss bursts as the likely first investigation path.
- Enable the promotion price cache single-flight lock.
- Lower promotion price cache TTL to 60 seconds during rollout.
- Keep payment writes enabled unless payment timeout error rate exceeds 12% for 5 minutes.
- Do not restart checkout pods unless there is crash-loop or memory-pressure evidence.
- Do not truncate or drop promotion cache tables during live traffic.
""".strip())

(dsh_workspace / "prior_incident.md").write_text("""
Prior lesson:
- Avoid restarting all checkout pods without crash-loop evidence.
- Previous incident was mitigated by single-flight cache miss protection plus short TTL during rollout.
""".strip())

prompt = (
    "Read incident_ticket.md, runbook.md, and prior_incident.md. "
    "Produce an incident response plan with likely_cause, evidence, safe_next_action, "
    "rollback_plan, customer_impact, and open_questions. Do not invent tools, owners, "
    "dashboards, thresholds, or operational facts that are not in the files."
)

print("Workspace prepared:", dsh_workspace)
print("Running through Ollama launcher with model:", DEEPSEEK_DSH_MODEL)

!cd /content/dsh-incident-demo && timeout 240s ollama launch dsh --model "$DEEPSEEK_DSH_MODEL" -- --profile headless "$prompt" || true


In [ ]:
DEEPSEEK_OUTPUT = """
Paste DeepSeek Harness incident-response output here if you run it separately.
""".strip()

if DEEPSEEK_OUTPUT and "Paste DeepSeek" not in DEEPSEEK_OUTPUT:
    deepseek_scored = score_freeform_answer(
        scenario=scenario,
        answer=DEEPSEEK_OUTPUT,
        lane=Lane.DEEPSEEK_PROVIDER,
        title="DeepSeek Harness output experiment",
        takeaway="DeepSeek Harness represents the plug-and-play runtime direction for harness engineering.",
        used_harness_memory=True,
    )
    display(Markdown(render_management_summary_markdown(deepseek_scored)))
    display(Markdown(render_executive_findings_markdown(deepseek_scored)))
    display(Markdown(render_rule_findings_markdown(scenario, deepseek_scored)))
    display(HTML(render_model_output_html("DeepSeek Harness output", deepseek_scored.final_answer)))

    deepseek_findings = evaluate_rules(scenario, deepseek_scored)
    deepseek_summary = summarize_findings_with_ollama(
        scenario=scenario,
        result=deepseek_scored,
        findings=deepseek_findings,
        model_name=SUMMARY_MODEL,
    )
    display(Markdown("# DeepSeek Harness: LLM-Polished Summary"))
    display(Markdown(deepseek_summary))
else:
    display(Markdown("""
## DeepSeek Harness Output Not Scored Yet

No DeepSeek Harness agent output was provided in `DEEPSEEK_OUTPUT`.

For the management narrative, say:

> DeepSeek Harness shows the plug-and-play runtime direction. If it does not run cleanly in Colab today, that is a maturity/setup finding about the framework path, not a failure of harness engineering. The live Ollama no/weak/strong lanes already proved the harness value.
"""))

# Closing Narrative

The medium model is not magically smarter. It performs better because the harness gives it:

- controlled context
- tool boundaries
- shared memory
- policy/runbook grounding
- reviewer checks
- objective sensors
- repair loop
- repeatable scorecard

That is the difference between a chat answer and a production AI workflow.